# Visualising data with Seaborn

[Seaborn](http://seaborn.pydata.org/).

This week is about turning a research question into a clear graph. We will use a few useful graph types. Bar graphs, scatter plots, line plots.


Seaborn is an amazing tool that defines simple high-level functions for common statistical plot types, and integrates with the functionality provided by Pandas DataFrames.

For more information on plotting with Seaborn, see the [Seaborn documentation](http://seaborn.pydata.org/), a [tutorial](http://seaborn.pydata.org/
tutorial.htm), and the [Seaborn gallery](http://seaborn.pydata.org/examples/index.html).

## Setup

Seaborn makes graphs from pandas DataFrames. Matplotlib gives us control over the figure, labels and scales.

In [ ]:
# Store tables and perform selections, grouping and merging with pandas.
import pandas as pd
# Use Matplotlib to create figures and customise axes.
import matplotlib.pyplot as plt
# Use Seaborn to map DataFrame columns onto plot elements.
import seaborn as sns

# Set the default appearance for subsequent plots.
sns.set_theme(style="whitegrid")

In [ ]:
# These files were introduced in Week 3.
# Read the CSV into a DataFrame; ../ moves up from week_5 to the course folder.
scotland_names_df = pd.read_csv("../week_3/scotland_babynames.csv")
# Load US records, which include a region column.
usa_names_df = pd.read_csv("../week_3/us_babynames_by_region.csv")

# head() previews the first five rows without changing the table.
display(scotland_names_df.head())
display(usa_names_df.head())

The `_df` ending is a useful label for us: it reminds us that these variables contain whole DataFrames.

Before plotting, say what one row represents. In the Scottish table, one row is one name, for one sex, in one year. In the US table, one row also belongs to one region.

## Bar charts compare categories

**Question:** What were the ten most common girls' names registered in Scotland in 2025?

First make the small table that the graph will display. A graph is not a substitute for preparing the right data.

In [ ]:
# Create a Boolean mask: True for rows whose year is 2025.
select_2025 = scotland_names_df["Year"] == 2025
# Create a second mask for records labelled F.
select_girls = scotland_names_df["Sex"] == "F"

# loc selects rows matching BOTH masks (&), then keeps the listed columns.
girls_2025_df = scotland_names_df.loc[
    select_2025 & select_girls,
    ["Name", "Count"],
]

# Sort largest counts first, then keep the first ten rows.
# Display the prepared table so we can check what the graph will show.
top_10_girls_df = girls_2025_df.sort_values("Count", ascending=False).head(10)
# Display the prepared table so we can check what the graph will show.
top_10_girls_df

## How Seaborn builds a graph

Seaborn makes graphs directly from a pandas DataFrame. We tell it **which table to use** and **which job each column should perform**:

```python
sns.PLOTTYPE(
    data=df,
    x="column_name",
    y="column_name",
    hue="column_name",
)
```


- `data=` specifies the DataFrame containing the data.
- `x=` selects the column placed along the horizontal axis.
- `y=` selects the column placed along the vertical axis.
- `hue=` uses another column to separate groups by colour.

A useful way to read this code aloud is:

The column names are written as strings because Seaborn looks for columns with those names inside the DataFrame supplied to `data=`.


Quick note what I wrote above python considers -identical- to this: 

```python
sns.PLOTTYPE( data=df, x="column_name", y="column_name", hue="column_name", )
```


It's a function call with multiple inputs.   I put each input on a separate line so the they are easier to read and change.  

# Seaborn bar graph

Next we'll turn a table into a dataframe.  

In [ ]:
# this line initializses a figure.   it is a good idea to do this before plotting, 
# so that you can control the size of the figure, and have a convienient 
# reference to the figure and axes objects that can be used to customize the plot.
fig, ax = plt.subplots(figsize=(8, 5))

# Draw one horizontal bar per name from the prepared table.
sns.barplot(
    # data chooses the table; x and y identify its columns.
    data=top_10_girls_df,
    x="Count",
    y="Name",
    # Use one colour because colour is not encoding an additional variable.
    color="#6A5ACD",
    # Draw into the axes created above.
    ax=ax,
)


# #Here we are using the axes object "ax" to set the title and labels for the plot. 
# ax.set_title("Ten most common girls' names in Scotland, 2025")
# ax.set_xlabel("Number of babies")
# ax.set_ylabel("Name")
# ax.set_xlim(left=0)

# Display the figure. The commented label commands can be enabled during class.
plt.show()

> **Teaching note**
>
> **Live-coding note:** The title and axis-label commands are commented out. Uncomment them to see how labels improve the graph. `barplot()` normally averages repeated observations in a category; here each name already has one count.

`fig` is the whole picture. `ax` is the plotting area—the part with the axes, labels and data. We pass `ax=ax` to Seaborn, then use the same object to improve the graph.

A horizontal bar chart leaves room for the names. The bars start at zero because their lengths represent amounts.

### Live coding pause: extend the question

Change the selections to show boys' names, or choose another year. Before running the graph, predict one thing that might change.

## Scatter plots show the relationship between two numeric variables

**Question:** In the 2025 Scottish data, Does the length of a name relate to a babies gender?

We need to derive a new numeric column. The string method `.str.len()` counts the characters in every name.

In [ ]:
# Select 2025 and make an independent copy before adding a new column.
names_2025_df = scotland_names_df.loc[select_2025].copy()
# str.len() counts characters in each name, creating one value per row.
names_2025_df["NameLength"] = names_2025_df["Name"].str.len()

# Inspect the selected columns to check the new values.
names_2025_df[["Sex", "Name", "NameLength", "Count"]].head()

In [ ]:
# Create the figure and the axes we will label below.
fig, ax = plt.subplots(figsize=(8, 5))

# Draw one point for each name/sex record.
sns.scatterplot(
    data=names_2025_df,
    x="NameLength",
    y="Count",
    # Use a separate colour for each recorded sex.
    hue="Sex",
    # Make points partly transparent so overlaps are easier to see.
    alpha=0.55,
    ax=ax,
)

# Give the figure context and replace column names with readable axis labels.
ax.set_title("Name length and popularity in Scotland, 2025")
ax.set_xlabel("Number of characters in the name")
ax.set_ylabel("Number of babies")
# Explain what the colours represent.
ax.legend(title="Sex recorded at registration")

# Display the completed figure.
plt.show()

> **Teaching note**
>
> **Watch the unit of observation:** Each point is a name/sex record, not an individual baby. A rare name and a common name each contribute one point. This helps explain why this plot does not directly answer the question about name length and sex.

Each point is one name. `hue` gives a categorical variable the job of controlling colour. Transparency helps us see where points overlap.

This isn't very good way to look at this data.  Every different name is a different dot! But we didn't care about different names, we grouped them together.   

### Not every plot is a useful way to investigate the data

This plot didn't do a great job addressing that question.    We can come back to that. 

## Counts are not always comparable

Suppose 1,000 girls are represented in one place and 10,000 in another. A count of 50 has a very different meaning in those two populations.  Population has grown a lot over the last 100 years so it's usefule to consider the percentage of babies given a name, not just the count.   

But instead of per 100 (like percent) I'm going to do it as  popularity **per 1,000 recorded names**.   It's the same concept with different unit.   


This is the same general idea as calculating cases per 100,000 in the assignment.

In [ ]:

# Group all names by year and sex, then add their counts for each group.
scotland_total_df = (
    # as_index=False keeps Year and Sex as ordinary columns for joining later.
    scotland_names_df.groupby(["Year", "Sex"], as_index=False)["Count"]
    .sum()
)

#Let's rename that column to "TotalCount" so that we can merge it with the original dataframe later.
scotland_total_df.rename(columns={"Count": "TotalCount"}, inplace=True)

# Plot one line per sex; estimator=None uses the totals already calculated.
sns.lineplot(data=scotland_total_df, x="Year", y="TotalCount", hue="Sex", estimator=None)

Once we have calculated those we can merge them back into the main dataframe. 

In [ ]:
#Merge the total counts back in.  Note that the merge is needed because
#the grouped counts collapsed all the names for each year and sex into a single row, so we need to merge it back to the original dataframe to have the total counts available for each name.
# Match on BOTH year and sex. A left join keeps all original name rows.
scotland_names_df= pd.merge(scotland_names_df, scotland_total_df, on=["Year", "Sex"],how="left")

> **Teaching note**
>
> **Watch out when rerunning cells:** This merge overwrites `scotland_names_df`. Running it again can create `TotalCount_x` and `TotalCount_y` instead of `TotalCount`, breaking the next calculation. Reload the original data and run the preparation in order. Duplicate keys in a totals table can also multiply rows in a merge.

In [ ]:
# Divide each name count by its matching yearly/sex total; multiply by 1,000 for the units.
scotland_names_df["CountPer1000"] = 1000 * (scotland_names_df["Count"] / scotland_names_df["TotalCount"])
scotland_names_df

In [ ]:
#Let's check George. 
# Keep records with this exact spelling; this selection does not restrict sex.
select_name = scotland_names_df["Name"] == "George"

# Plot the selected records. This example still uses raw Count, not CountPer1000.
sns.lineplot(
    data=scotland_names_df[select_name],
    x="Year",
    y="Count"
)

> **Teaching note**
>
> **Check what is being plotted:** This uses `Count`, although a rate was just calculated. It also selects George without selecting a sex. If a year has multiple matching records, Seaborn averages them by default. Decide which measure and group answer the question before interpreting the line.

##

### Now let's do this for the USA

In [ ]:

#Calculate the totals for year, region, sex seperately. 
# Calculate a separate denominator for every combination of year, region and sex.
totals_by_region_df = (
    usa_names_df
    # Keep the group identifiers as columns, and select Count to sum.
    .groupby(["Year", "Region", "Sex"], as_index=False)["Count"]
    .sum()
    # Give the denominator a different name so it is distinct from each name count.
    .rename(columns={"Count": "TotalNamesByGender"})
)

#Merge the totals back in. 
# Attach each total to name rows with the same three identifiers.
usa_names_df = pd.merge(
    usa_names_df,
    totals_by_region_df,
    on=["Year", "Region", "Sex"],
)

#Now add in the CountPer1000 column to the usa_names_df dataframe.
# Convert each count into a rate using its own year, region and sex denominator.
usa_names_df["CountPer1000"] = 1000 * (usa_names_df["Count"] / usa_names_df["TotalNamesByGender"])
# Check the new total and rate columns.
usa_names_df.head()

> **Teaching note**
>
> **Watch out when rerunning cells:** This also overwrites the loaded DataFrame. Repeating the merge can produce suffixed total columns. Each year/region/sex combination should match exactly one denominator.

The work happens in visible stages:

- `groupby()` calculates the relevant denominator for every year and region and sex.
- `merge()` attaches the correct denominator to each count.
- The formula creates the comparable measure.

Always inspect the prepared table before trusting the graph.

### Now let's look at the popularity over time for a spec

In [ ]:
# Create a name-selection mask aligned to usa_names_df.
select_hillary = usa_names_df["Name"] == "Hillary"

#Notice the boolean & takin gboth Hillary and the years. 
# Combine both masks with &, keep all columns (:), and preview ten rows.
usa_names_df.loc[select_hillary , :].head(10)

In [ ]:
# hillary_recent_df = hillary_df.loc[hillary_df["Year"] >= 1960]
# This mask currently selects Taylor, despite the variable name and labels below.
select_hillary = usa_names_df["Name"] == "Hillary"

# Create a new plotting area.
fig, ax = plt.subplots(figsize=(10, 5))

# Without hue, Seaborn combines the rows at each year using its default mean.
sns.lineplot(
    data=usa_names_df.loc[select_hillary], #<--- We can pass in a subset of a dataframe.
    x="Year",
    y="CountPer1000",
    hue="Region", #<--- This will create a separate line for each region.
    ax=ax,
)

# Draw a reference line at the chosen x value; label supplies its legend text.
ax.axvline(1992, color="black", linestyle="--", label="1992 election")
# Set the title and units: these must agree with the selected data.
ax.set_title("Popularity of Hillary in US regions")
ax.set_xlabel("Year")
ax.set_ylabel("Babies named Hillary per 1,000 recorded girls' names")
# Start the rate axis at zero.
ax.set_ylim(bottom=0)
# ax.legend(title="Region / event")

# Display the figure; enable ax.legend() above to show the event label.
plt.show()

The name peaks in 1992 and falls sharply afterwards in every region. The timing is interesting and worth discussing, but the graph alone does not prove that the election caused the change.

Note about the error bars.   Seaborn nicely often adds in errorbars for for you.  In this case we have multiple regions and it calculates the variability for you. 

Note: Check counts versus comparable rates

Change `y="CountPer1000"` to `y="Count"`. Which regional differences become larger? Why might population size be influencing that graph?

## Now lets return to the name lengths

We should calculate the totals ourselves.  

In [ ]:

#Calculate the total for the count. 


# calcaulte the total for count, and keep the name length and sex columns.  This will give us a dataframe with the total counts for each name length and sex combination.
# Add babies across names of the same length and sex, giving one row per combination.
names_total_by_length_2025_df = names_2025_df.groupby(["NameLength", "Sex"], as_index=False)["Count"].sum()

names_total_by_length_2025_df


In [ ]:
# Plot the already-summed counts: one bar per name-length/sex combination.
sns.barplot(
    data=names_total_by_length_2025_df,
    x="NameLength",
    y="Count",
    # Separate the two sex categories by colour within each name length.
    hue="Sex",

)

> **Teaching note**
>
> **Why group first?** Summing `Count` counts babies at each name length. Without this preparation, `barplot()` would show the mean count per name instead. Check what one row represents before choosing a plot.



## Questions to ask about a graph

- What does one point, bar or line represent?
- Are the title, labels, units and legend understandable and legible?
- Does the graph answer the stated question?
- If two graphs are compared, are their scales comparable?
- Does a quick data audit table agree with the graph?
- Are colors/labels used consistently?
